<a href="https://colab.research.google.com/github/UnitForDataScience/Intro_to_TimeSeries/blob/main/Don't_Break_the_Timeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Don't Break the Timeline: An Intro to Time Series Forecasting
## Hands-On Exercise: Forecasting Energy Usage

---

####An Open Lab by the UDS at ASU Library


Welcome! This notebook is part of our introductory webinar on time series forecasting. It guides you through a use case that demonstrates the need for time series analysis in forecasting household energy usage over an entire year.

In the following sections, we will load an external dataset, explore it to gain a better understanding, and preprocess it to suit our needs. Next, we will use the classic time series analysis method, ARIMA, as our baseline model to forecast future household energy use. Finally, we will utilize a more modern time series algorithm called Prophet and compare our results.

By the end of this notebook, you will have a beginner-level understanding of how to implement time series analysis and how to select a suitable method based on your data.

The Dataset: UCI Household Electric Power Consumption.

The Goal: Train on 2006–2009 data -> Predict 2010 usage.

##Installing the necessary libraries

When working with Python, the first step is to load the necessary libraries and download important packages. These libraries are free resources that provide essential functionalities for data manipulation and time series analysis. In the cell below, we will install several libraries that will be utilized throughout this notebook. Although the installation might take some time in Colab, it will be worth the wait. The comments (in green) explain the capabilities each library adds to our code.

In [ ]:
# Working with tables
import pandas as pd

# Data Visualization
import matplotlib.pyplot as plt

# Numerical Calculations
import numpy as np

# Accessing system parameters
import sys

# Install pmdarima library
!{sys.executable} -m pip install pmdarima
import pmdarima as pm

print("pmdarima installed successfully.")

# ARIMA
from statsmodels.tsa.arima.model import ARIMA

# Install prophet library
!{sys.executable} -m pip install prophet

print("Prophet installed successfully.")

# Import Prophet
from prophet import Prophet

# Silence warnings
import warnings
warnings.filterwarnings("ignore")

## Selecting and Loading a Dataset

One of the most crucial steps at the onset of your time series analysis is choosing a suitable dataset. Datasets used for time series analysis should include a clear timestamp column and a numerical value that you are interested in forecasting. For this short demo, we will use the '[Individual Household Electric Power Consumption](https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption.)' dataset from the UC Irvine's Machine Learning repository.

This dataset consists of electric power consumption measurements in one household located in Sceaux (7km of Paris, France). The data, with a one-minute sampling rate, was gathered between December 2006 and November 2010 (47 months), for a total of 2075259 measurements. This archive contains different electrical quantities, and some sub-metering values are available.

To add this dataset to our notebook, we will do the following:

1. Download the dataset as a `.zip` file, and unzip the downloaded file. This will result in a `.txt` file containing the data.
2. Load the data into a Pandas DataFrame to use with Python. The `.txt` file uses a delimiter (';') to separate text that should be in different columns.
3. Prepare a single column that contains the date and time data (this will facilitate things later on)





In [ ]:

# Downloading the Dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
!wget -q -O /tmp/hpc.zip {url}
!unzip -q /tmp/hpc.zip -d /tmp/hpc

#Red the downloaded file and turn it into a "DataFrame"
df = pd.read_csv("/tmp/hpc/household_power_consumption.txt", sep=';', na_values=['?'],low_memory=False)

# Combine 'Date' and 'Time' columns into a single datetime series and set as index
df['dt'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S', dayfirst=True, errors='coerce')
df = df.set_index('dt').sort_index()

# Drop the original 'Date' and 'Time' columns as they are now combined into the index
df = df.drop(columns=['Date', 'Time'])

# Convert remaining columns to numeric type, coercing errors to NaN
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("\nDataset loaded successfully. Displaying the first 10 rows:")
print(df.head(10))

print("\nDataFrame Info:")
df.info()

## Initial Data Exploration and Preprocessing

With our data loaded into the notebook, we can begin exploring and preprocessing our dataset. We will identify the time period covered by the dataset, check for missing values (and replace any missing data with estimates), and "resample" the data to a daily frequency.


In [ ]:
# 1. Determine the start and end dates of the dataset
start_date = df.index.min()
end_date = df.index.max()
print(f"\nDataset start date: {start_date}")
print(f"Dataset end date: {end_date}")

# 2. Check for missing values
print("\nMissing values in DataFrame (df):\n")
print(df.isnull().sum())

# 3. Resample the DataFrame to a daily frequency; we will focus on the daily mean
# value of 'Global_active_power' for forecasting.
df_daily = df.resample('D').mean()

print("\nDaily resampled DataFrame (df_daily) head:")
print(df_daily.head())

# 4. Check for any remaining missing values in df_daily and impute
print("\nMissing values in daily resampled DataFrame (df_daily) before imputation:\n")
print(df_daily.isnull().sum())

# Impute missing values for 'Global_active_power' using forward-fill then backward-fill
# This ensures that any gaps are filled by the nearest valid observation.
df_daily['Global_active_power'] = df_daily['Global_active_power'].ffill().bfill()


print("\nDaily resampled DataFrame (df_daily) info after imputation:")
df_daily.info()

## Exploratory Data Analysis (EDA) and Decomposition

Let’s continue our exploration of the resampled dataset by first visualizing the raw time series. After that, we will decompose it into its trend, seasonal, and residual components. This process will help us visualize each component individually, enhancing our understanding of their roles and related concepts, such as stationarity.


In [ ]:
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

# Select the 'Global_active_power' column for plotting and decomposition
time_series_data = df_daily['Global_active_power']

# 1. Visualize the raw time series
plt.figure(figsize=(14, 7))
plt.plot(time_series_data)
plt.title('Daily Global Active Power Over Time')
plt.xlabel('Date')
plt.ylabel('Global Active Power (kW)')
plt.grid(True)
plt.show()

# 2. Perform additive seasonal decomposition
# We will choose a period of 365 to represent yearly seasonality since we have daily data.
# The model is 'additive' as it's common for power consumption where fluctuations tend to be constant over time.
decomposition = seasonal_decompose(time_series_data, model='additive', period=365, extrapolate_trend='freq')

# 3. Plot the decomposed components
fig = decomposition.plot()
fig.set_size_inches(11.7, 9)
fig.suptitle('Time Series Decomposition of Daily Global Active Power', y=1.02) # Adjust suptitle position

plt.subplot(4, 1, 1).set_title('Original Series', loc='left')
plt.subplot(4, 1, 2).set_title('Trend Component', loc='left')
plt.subplot(4, 1, 3).set_title('Seasonal Component', loc='left')
plt.subplot(4, 1, 4).set_title('Residual Component', loc='left')

plt.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust layout to prevent overlap
plt.show()

### Explanation of Time Series Decomposition Components

Time series decomposition separates a time series into several components, typically trend, seasonality, and residuals, to better understand its underlying structure. For the 'Daily Global Active Power Over Time' dataset, these components represent:

*   **Original Series**: This is the raw time series data (`Global_active_power`) plotted against time, showing the overall pattern including all variations.

*   **Trend Component**: This component captures the long-term progression or direction of the series. It smooths out short-term fluctuations and seasonality to reveal whether the global active power is generally increasing, decreasing, or remaining stable over the years. In our plot, the trend seems to show an initial high usage, followed by a slight decrease, and then remaining relatively stable with some fluctuations.

*   **Seasonal Component**: This component represents patterns that repeat over a fixed period. Since we chose a `period` of 365 days (yearly seasonality), this component illustrates how the global active power usage typically changes throughout a year. For example, we can observe higher usage during certain months or seasons and lower usage during others. The seasonal plot shows a clear yearly cycle, indicating higher power consumption during colder months (winter) and lower during warmer months (summer).

*   **Residual Component**: Also known as the 'noise' or 'error' component, this represents the random, unpredictable variations in the time series after the trend and seasonal components have been removed. It's what's left over and ideally should show no discernible pattern, indicating that the trend and seasonality have successfully captured the systematic movements in the data.

### Concept of Stationarity and its Relevance

**Stationarity** is a fundamental concept in time series analysis. A time series is considered stationary if its statistical properties—like mean, variance, and autocorrelation—remain constant over time. In simpler terms, a stationary series looks much the same no matter when you observe it; there are no systematic changes in its properties over time.

There are different types of stationarity, with **strict stationarity** (all statistical properties constant) and **weak-sense stationarity** (mean, variance, and autocovariance constant) being common. For most practical applications in time series modeling, weak-sense stationarity is usually sufficient.

**Why is stationarity important for traditional time series models like ARIMA?**

Many traditional time series models, particularly **ARIMA (AutoRegressive Integrated Moving Average)** models, assume that the underlying time series is stationary. The reasons for this are:

1.  **Predictability**: If a series is stationary, we can assume that its future statistical properties will be similar to its past properties. This predictability is crucial for making reliable forecasts.

2.  **Model Validity**: The mathematical foundations of ARIMA models are based on the assumption of stationarity. For instance, the coefficients of AR (AutoRegressive) and MA (Moving Average) components in ARIMA models are meaningful only when the series is stationary. If the series is non-stationary, these coefficients can be unstable and lead to incorrect inferences and forecasts.

3.  **Statistical Inference**: Statistical tests and confidence intervals for model parameters are valid only when the underlying data is stationary. Non-stationary data can lead to spurious regressions, where two unrelated time series appear to be correlated due to common trends.

4.  **Differencing**: If a time series is non-stationary (e.g., it has a trend), it can often be made stationary by applying a differencing operation (subtracting the previous observation from the current one). This is where the 'I' (Integrated) in ARIMA comes from, indicating the differencing required to achieve stationarity. Once differenced, the ARIMA model can then be applied to the stationary series.

In essence, ensuring stationarity (or transforming a non-stationary series into a stationary one) is a critical preprocessing step for ARIMA models to yield accurate and reliable results.

# Time Series Forecasting with ARIMA

"This is where the fun begins" -Anakin Skywalker

## Splitting the dataset for ARIMA

We're not done with the dataset! Before jumping headfirst in into your first forecast, we need to divide the dataset into a "training" and testing sets.
We will split the time series dataset, using data from 2006-2009 for training and the year 2010 for testing and forecasting. This ensures the ARIMA model is trained on a specific period and evaluated on the subsequent year, helping us determine how well it performs.


In [ ]:
# Define the split dates
train_end_date = '2009-12-31'
test_start_date = '2010-01-01'

# Adjust the time series data split
train_data = time_series_data.loc[time_series_data.index <= train_end_date]
test_data = time_series_data.loc[time_series_data.index >= test_start_date]

print(f"New Training data length: {len(train_data)}")
print(f"New Testing data length: {len(test_data)}")

print("\nNew Train data head:")
print(train_data.head())
print("\nNew Test data head:")
print(test_data.head())

## Introduction to ARIMA Model

Remember, **ARIMA** stands for **AutoRegressive Integrated Moving Average**. It is a widely used statistical model for time series forecasting. The model is composed of three parts:

1.  **AR (AutoRegressive) - p**: This component refers to the use of past values in the regression equation for the current value. An AR(p) model predicts the future value based on a linear combination of `p` past observations. For example, if `p=1`, the current value depends on the immediate past value.

2.  **I (Integrated) - d**: This component represents the differencing order. Differencing involves subtracting the current value from a previous value in the series. The purpose of differencing is to make the time series **stationary**, which is a crucial assumption for ARIMA models. A stationary series has statistical properties (like mean, variance, and autocorrelation) that do not change over time. If `d=0`, no differencing is applied. If `d=1`, the series is differenced once, meaning we model the changes between consecutive values.

3.  **MA (Moving Average) - q**: This component incorporates the dependency between an observation and a residual error from a moving average model applied to lagged observations. An MA(q) model uses `q` past forecast errors in the regression. For example, if `q=1`, the current value depends on the immediate past error term.

Together, the parameters (p, d, q) define the ARIMA model order:
*   **p**: The order of the AutoRegressive (AR) part.
*   **d**: The order of differencing (I) part.
*   **q**: The order of the Moving Average (MA) part.

##ARIMA Parameter Selection
Choosing the ARIMA parameters is key to obtaining meaningful results. There are several methods to help you do so, in this notebook, we will use the "auto_arima" fuction to help us find the best parameter combination for our use case. Notice how this function takes the "training" dataset we created to find the optimal parameters.

In [ ]:
# Use auto_arima to find optimal (p, d, q) parameters
# Suppress warnings to keep output clean


print("Searching for optimal ARIMA parameters using auto_arima...")
stepwise_fit = pm.auto_arima(train_data, start_p=1, start_q=1,
                             max_p=7, max_q=7, m=1, # m=1 for daily data (no seasonality considered by default for non-seasonal ARIMA)
                             start_P=0, seasonal=False, # We are building a non-seasonal ARIMA for now
                             d=None, trace=True,
                             error_action='ignore',   # don't want to know if a particular model fails
                             suppress_warnings=True,  # don't want convergence warnings
                             stepwise=True)

print("\nOptimal ARIMA Model Summary:")
print(stepwise_fit.summary())

# Extract the optimal (p, d, q) parameters
arima_order = stepwise_fit.order
print(f"\nOptimal ARIMA Order (p, d, q): {arima_order}")

#### Rationale for ARIMA(3,1,3) Parameters

- **p = 3 (AR order)**: This suggests that the current value of the time series can be predicted using its previous 3 values. The `auto_arima` function found that including these many past observations helps in modeling the patterns and dependencies within the data.

- **d = 1 (Differencing order)**: This indicates that the time series needed to be differenced once to achieve stationarity. This aligns with common practices where many real-world time series, especially those with trends, are non-stationary and require first-order differencing.

- **q = 3 (MA order)**: This implies that the current value can also be explained by the error terms of the previous 3 forecast errors. Including these past error terms helps capture short-term fluctuations that the autoregressive components may miss.


## The ARIMA Forecast

Now that we have our parameters, we can go ahead and train the model.Once the model is trained, we will generate a forecast for the "test" period (i.e. the year 2010), genrate confidence intervals for reference, and then plot the forecast against the real data.


In [ ]:
# (We already imported these, I leave them commented here in case you want to copy/paste this cell)
#import numpy as np
#from statsmodels.tsa.arima.model import ARIMA # Added import statement
#import matplotlib.pyplot as plt

# Training the model
model = ARIMA(train_data, order=arima_order)
model_fit = model.fit()
print("ARIMA model trained successfully.")
# print(model_fit.summary()) # Uncomment to see the full summary of the fitted model

# Get the start and end dates for the forecast period
start_index = test_data.index.min()
end_index = test_data.index.max()

# Generate forecasts for the test period
print("\nGenerating forecasts...")
# The get_forecast method requires start and end dates, or steps
# In our case, we will forecast for the length of the test data
forecast_result = model_fit.get_forecast(steps=len(test_data))

# Extract predicted means and their confidence intervals
forecast_mean = forecast_result.predicted_mean
conf_int = forecast_result.conf_int(alpha=0.05) # 95% confidence interval

# Align the forecast index with the test data index
forecast_mean.index = test_data.index
conf_int.index = test_data.index

print("Forecasts generated.")


##Plotting our ARIMA Forecast

Now that our model is fitted, we can proceed to plot the forecast and compare it to the original daily values for "Global Active Power"

In [ ]:
print("""Overall Purpose of this Notebook:\nThis notebook serves as an introduction to time series forecasting, demonstrating a complete workflow from data loading and preprocessing to model training and evaluation. It specifically explores two popular forecasting methods, ARIMA and Prophet, using the 'Individual Household Electric Power Consumption' dataset. The aim is to teach students how to perform exploratory data analysis, understand time series decomposition, implement and compare traditional statistical models (ARIMA) with more modern machine learning-based approaches (Prophet), and interpret their results.\n""")

# Plot the original train_data, test_data, and ARIMA forecasts
plt.figure(figsize=(16, 8))
plt.plot(train_data.index, train_data, label='Training Data', color='blue')
plt.plot(test_data.index, test_data, label='Actual Test Data', color='green')
plt.plot(forecast_mean.index, forecast_mean, label='ARIMA Forecast', color='red', linestyle='--')
plt.fill_between(conf_int.index, conf_int.iloc[:, 0], conf_int.iloc[:, 1], color='pink', alpha=0.3, label='95% Confidence Interval')

plt.title(f'ARIMA{arima_order} Forecast vs Actuals')
plt.xlabel('Date')
plt.ylabel('Global Active Power (kW)')
plt.legend()
plt.grid(True)
plt.show()

# Time Series Forecasting with Prophet

"Lisan al Gaib!" - Stilgar


### Introduction to Prophet Model

**Prophet** is an open-source forecasting library developed by Facebook, specifically designed for forecasting time series data with strong seasonal effects and several seasons of historical data. It is particularly robust to missing data and shifts in the trend, and typically handles outliers well. Prophet's model is an additive regression model with four main components:

1.  **A piecewise linear or logistic growth curve trend.** Prophet automatically detects changes in trend by selecting changepoints from the data.
2.  **A yearly seasonal component** modeled using Fourier series.
3.  **A weekly seasonal component** modeled using dummy variables.
4.  **A daily seasonal component** modeled using Fourier series.

### Advantages of Prophet for Time Series Forecasting:

*   **Intuitive Parameters**: You can easily configure different types of seasonality (yearly, weekly, daily), add custom holidays, and define changepoints.
*   **Handles Seasonality Effectively**: Prophet excels at modeling multiple periods of seasonality (e.g., daily, weekly, yearly patterns) simultaneously, which is common in many business and consumption time series. Under the hood, it uses a Fourier series to model these periodic effects.
*   **Robust to Missing Data and Outliers**: Prophet can handle missing data and is designed to be robust to anomalous data points.
*   **Flexible Holiday Modeling**: It allows for easy incorporation of custom holiday effects. You can provide a list of past and future holidays, and Prophet will treat them as special events in the forecast.
*   **Automatic Trend Detection**: Prophet automatically detects `changepoints` in the time series, allowing for flexible modeling of non-linear trends over time without manual intervention.
*   **Scalability**: It's designed to work well with large datasets and can be easily automated for many time series at once.


### Preprocessing data for Prophet
To begin, let's prepare the `train_data` and `test_data` by resetting their index and renaming the columns to 'ds' for the datetime index and 'y' for the 'Global_active_power' column. This is the format required by Prophet.



In [ ]:
# Prepare training data for Prophet
prophet_train_df = pd.DataFrame(train_data).reset_index()
prophet_train_df.rename(columns={'dt': 'ds', 'Global_active_power': 'y'}, inplace=True)

# Prepare testing data for Prophet (only 'ds' is needed for future dataframe, 'y' for actual comparison)
prophet_test_df = pd.DataFrame(test_data).reset_index()
prophet_test_df.rename(columns={'dt': 'ds', 'Global_active_power': 'y'}, inplace=True)

print("Prophet training data head:")
print(prophet_train_df.head())
print("\nProphet testing data head:")
print(prophet_test_df.head())

**Reasoning**:
Before instantiating and fitting the Prophet model, I need to ensure the `prophet` library is installed. After installation, I will import `Prophet`, instantiate it with specified seasonality settings (`yearly_seasonality=True`, `weekly_seasonality=True`, `daily_seasonality=False` as the data is daily aggregated), and then fit the model to the `prophet_train_df` data, covering instructions #3, #4, and #5 of the subtask.



In [ ]:
# Instantiate a Prophet model
# Set daily_seasonality=False as data is daily aggregated, so daily seasonality won't be distinct within a day.
# Set yearly_seasonality=True as our EDA showed clear yearly patterns.
# Set weekly_seasonality=True to capture weekly patterns that might exist in daily data.
model_prophet = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)

# Fit the Prophet model to the prepared training data
print("\nFitting Prophet model...")
model_prophet.fit(prophet_train_df)
print("\nProphet model fitted successfully.\n")

# We need to create a DataFrame with dates to make predictions for.
# We want to forecast for the test period (all of 2010).
# The test_data has 330 entries, so we need to generate 330 future dates starting from the end of train_data.

future = model_prophet.make_future_dataframe(periods=len(prophet_test_df), freq='D', include_history=False)

print("Future DataFrame head:")
print(future.head())
print(f"Future DataFrame length: {len(future)}")

# Generate forecasts using the fitted Prophet model for the future dates we set up
print("\nGenerating Prophet forecasts...")
forecast = model_prophet.predict(future)
print("Prophet forecasts generated.")

print("\nProphet forecast head:")
print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].head())

# To see the full output of our forecast, uncomment the below
#print(forecast.head())

##Plotting our Prophet Forecast

Now that our model is fitted, we can proceed to plot the forecast and compare it to the original daily values for "Global Active Power"

In [ ]:
import matplotlib.pyplot as plt

# Plot the Prophet forecast, actual test data, and training data
plt.figure(figsize=(16, 8))
plt.plot(prophet_train_df['ds'], prophet_train_df['y'], label='Training Data', color='blue')
plt.plot(prophet_test_df['ds'], prophet_test_df['y'], label='Actual Test Data', color='green')
plt.plot(forecast['ds'], forecast['yhat'], label='Prophet Forecast', color='red', linestyle='--')
plt.fill_between(forecast['ds'], forecast['yhat_lower'], forecast['yhat_upper'], color='pink', alpha=0.3, label='95% Confidence Interval')

plt.title('Prophet Forecast vs Actuals')
plt.xlabel('Date')
plt.ylabel('Global Active Power (kW)')
plt.legend()
plt.grid(True)
plt.show()

# Comparing and Evaluating the Forecasts


We can evaluate the performance of both the ARIMA and Prophet models using appropriate metrics (e.g., RMSE, MAE) on a test set. We can also compare their forecasts visually and discuss the strengths and weaknesses of each model based on the results.


##Side-by-Side Forecasting: ARIMA and Prophet

In [ ]:
# Create a single plot for visual comparison
plt.figure(figsize=(14, 7))

# Plot actual test data
plt.plot(test_data.index, test_data, label='Actual Test Data', color='blue', alpha=0.7)

# Plot ARIMA forecast
plt.plot(forecast_mean.index, forecast_mean, label='ARIMA Forecast', color='red', linestyle='--')

# Plot Prophet forecast
plt.plot(forecast['ds'], forecast['yhat'], label='Prophet Forecast', color='green', linestyle='-.')

plt.title('ARIMA vs Prophet Forecast Comparison with Actuals', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Global Active Power (kW)', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# 1. Calculate RMSE and MAE for ARIMA model
# Ensure both test_data and forecast_mean have the same length and index for direct comparison
# Although forecast_mean already has the test_data index, a reindex can prevent issues if lengths differ slightly
actual_arima = test_data
predictions_arima = forecast_mean

rmse_arima = np.sqrt(mean_squared_error(actual_arima, predictions_arima))
mae_arima = mean_absolute_error(actual_arima, predictions_arima)

print(f"\nARIMA Model Performance:")
print(f"  RMSE: {rmse_arima:.4f}")
print(f"  MAE: {mae_arima:.4f}")

# 2. Calculate RMSE and MAE for Prophet model
# Ensure both prophet_test_df['y'] and forecast['yhat'] have the same length and index
actual_prophet = prophet_test_df['y']
predictions_prophet = forecast['yhat']

rmse_prophet = np.sqrt(mean_squared_error(actual_prophet, predictions_prophet))
mae_prophet = mean_absolute_error(actual_prophet, predictions_prophet)

print(f"\nProphet Model Performance:")
print(f"  RMSE: {rmse_prophet:.4f}")
print(f"  MAE: {mae_prophet:.4f}")

### Discussion of Model Strengths and Weaknesses

**Comparison and Analysis:**

1.  **Overall Fit and Accuracy:**
    *   **Prophet** outperformed ARIMA in terms of both RMSE and MAE. The lower values indicate that Prophet's forecasts were, on average, much closer to the actual values than ARIMA's.

2.  **Ability to Capture Patterns:**
    *   **ARIMA:** The visual plot shows that the ARIMA model does not capture the fluctuations and patterns in the test data, with a smoother, less responsive forecast line. However, given the seasonality of our data, this was to be expected. The non-seasonal ARIMA model, despite being tuned by `auto_arima`, is not adequately capturing the underlying seasonality or other complex patterns in the daily power consumption data.
    *   **Prophet:** The Prophet model's forecast line is visibly much closer to the actual test data. It appears to capture the daily variations and seasonal patterns more effectively. This is expected, as Prophet is specifically designed to handle multiple seasonalities (yearly, weekly, daily) and trends with changepoints, which are clearly present in this type of power consumption data. The confidence interval for Prophet also appears to contain most of the actual values, suggesting a good level of uncertainty estimation.

3.  **Over/Under-Forecasting:**
    *   **ARIMA:** Tends to exhibit both over and under-forecasting, especially failing to react sharply to sudden changes or sustained periods of lower/higher consumption. Its forecast appears somewhat flat compared to the actual data's variability.
    *   **Prophet:** While not perfect, Prophet's forecasts generally track the actual values much better, indicating less systematic over or under-forecasting. It adapts better to the changes in consumption levels throughout the year.

**In Sum: Strengths and Weaknesses:**

*   **ARIMA:**
    *   **Pros:** Good for stationary data or data that can be made stationary through differencing. Conceptually straightforward for understanding autoregressive and moving average components. Relatively fast to train for simpler models.
    *   **Cons:** Struggles with complex seasonality (especially multiple seasonal periods), and is sensitive to non-stationarity. Parameter selection (p, d, q) can be challenging! In this case, even an auto-selected non-seasonal ARIMA struggled with the clear yearly seasonality.

*   **Prophet:**
    *   **Pros:** Great at handling multiple seasonalities (yearly, weekly, daily), holidays, and trend changes automatically. Robust to missing data and outliers. Intuitive parameters and user-friendly API, making it accessible for non-experts. Provides clear component plots for interpretability.
    *   **Cons:** Can be computationally more intensive for very large datasets. It's a black-box model in some aspects compared to the statistical interpretability of ARIMA parameters.
